In [1]:
# LAB M3.03 Normal Objects - Creative Complaint Hander

In [ ]:
#Recommended fix added in new step to install the updated and more flexible packages
!pip -q install -U langchain langchain-core langchain-openai langchain-community

In [ ]:
#New code attempt: importing the needed libraries as well as OpenAI API key
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

print("LLM ready ✅")

In [ ]:
#Step 2 Create Creative Tools and define themed tools that the agent can use creatively.
# @tool is a decorator that turns turns a normal Python function into a LangChain Tool that an agent can call.

from langchain_core.tools import tool
import random
from typing import Optional

@tool
def consult_demogorgon(complaint: str) -> str:
    """Get the Demogorgon's perspective on a complaint about the Upside Down.

    Returns a creative, chaotic perspective that may be metaphorical or unsettling.
    """
    responses = [
        f"The Demogorgon tilts its head at '{complaint}'. Perhaps you're assuming Euclidean geometry applies here?",
        f"The Demogorgon emits a clicking sound that might be agreement. It suggests the issue is temporal—time behaves differently in the Upside Down.",
        f"The Demogorgon seems to be eating something. It does not understand '{complaint}'. Maybe consistency isn't a rule in that dimension."
    ]
    return random.choice(responses)


@tool
def check_hawkins_records(query: str) -> str:
    """Search Hawkins historical records for information on anomalous events.

    Returns a short 'record excerpt' if the query matches known topics.
    """
    records = {
        "portal": "Records show portals appear near spikes in electromagnetic activity and intense emotional events. Patterns are inconsistent but repeat in hotspots.",
        "monsters": "Records suggest Upside Down creatures shift behavior with proximity to gates, time of day, and environmental stressors.",
        "psychics": "Records indicate abilities vary by individual and are affected by fatigue, emotion, and focus. Capabilities are inconsistent under stress.",
        "electricity": "Hawkins has a long history of electrical anomalies correlated with gate activity and abnormal electromagnetic fields."
    }

    q = query.lower()
    for key, value in records.items():
        if key in q:
            return value

    return f"No direct record on '{query}'. Notes indicate unexplained events are common; recommend checking for EM interference and recent 'gate-adjacent' activity."


@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Suggest a creative interdimensional 'spell' to fix a problem.

    creativity_level: low, medium, high (defaults to medium). Unknown values treated as medium.
    Returns 1–3 imaginative suggestions.
    """
    level = (creativity_level or "medium").strip().lower()
    multiplier_map = {"low": 1, "medium": 2, "high": 3}
    creativity_multiplier = multiplier_map.get(level, 2)

    spells = [
        f"Chant 'BEMCA BECMA BECMA' three times while holding a Walkman to recalibrate frequencies linked to: {problem}",
        f"Draw a salt circle and place a compass in the center. Let magnetic drift stabilize: {problem}",
        f"Play 'Running Up That Hill' backwards at the problem site to induce temporal resonance for: {problem}",
        f"Gather: a lighter, a compass, and something personal. Arrange them in a triangle while focusing on: {problem}"
    ]

    selected = random.sample(spells, k=min(creativity_multiplier, len(spells)))
    return "\n".join(selected)


@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the D&D party (Mike, Dustin, Lucas, Will) for collective wisdom.

    Returns a collaborative, multi-voice response with practical hints.
    """
    party_responses = {
        "portal": (
            "Mike: 'Portals open near strong emotion or EM spikes.' "
            "Dustin: 'Also patterns cluster around repeat hotspots—look for 'gate-adjacent' signatures.'"
        ),
        "monsters": (
            "Lucas: 'They’re territorial but opportunistic.' "
            "Will: 'They sense fear and strong emotions. Staying calm changes outcomes.'"
        ),
        "psychics": (
            "Mike: 'Powers track emotional state.' "
            "Dustin: 'And energy limits. You can’t do everything at once.'"
        ),
        "electricity": (
            "Lucas: 'The Upside Down disrupts electrical systems.' "
            "Dustin: 'But it can also create weird feedback loops—watch for repeating flicker patterns.'"
        )
    }

    q = question.lower()
    for key, response in party_responses.items():
        if key in q:
            return response

    return (
        "The party huddles together. "
        "Mike: 'We need clearer details.' "
        "Dustin: 'Add time, location, and what changed recently.' "
        "Lucas: 'Look for patterns.' "
        "Will: 'If it feels like the Upside Down… trust that instinct.'"
    )


# Create list of tools
tools = [
    consult_demogorgon,
    check_hawkins_records,
    cast_interdimensional_spell,
    gather_party_wisdom
]

print(f"Created {len(tools)} creative tools:")
for t in tools:
    print(f"  - {t.name}: {t.description.splitlines()[0]}")

Created 4 creative tools:
  - consult_demogorgon: Get the Demogorgon's perspective on a complaint about the Upside Down.
  - check_hawkins_records: Search Hawkins historical records for information on anomalous events.
  - cast_interdimensional_spell: Suggest a creative interdimensional 'spell' to fix a problem.
  - gather_party_wisdom: Ask the D&D party (Mike, Dustin, Lucas, Will) for collective wisdom.


#Step 3 – Create Agent with Tools
In this step, we create a flexible LangChain agent using create_agent.
We provide: The LLM (llm) the list of creative tools (tools), a system prompt that encourages imaginative, tool-based problem solving. The agent can now decide autonomously which tools to call and in what order to resolve complaints in the Normal Objects universe.

In [7]:
from langchain.agents import create_agent

system_prompt = (
    "You are a creative complaint-resolution agent in the Normal Objects universe. "
    "You can consult tools (Demogorgon, Hawkins records, spells, party wisdom) in any order. "
    "Be imaginative but helpful. Always end with a clear final response."
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

print("Agent created ✅")

Agent created ✅


In [ ]:
#Step 3 – Invoke the Agent: send a user complaint to the agent using invoke() .The agent processes the request, optionally calls tools, and we print the final assistant response.

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "My basement portal flickers and my lights explode every Tuesday."}
    ]
})

print(result["messages"][-1].content)

It seems your basement portal issues are tied to both emotional spikes and electromagnetic activity, with historical records hinting that such anomalies often cluster in specific hotspots. The Demogorgon, in its chaotic way, suggests that perhaps the consistency of your Tuesdays is being misinterpreted—something is indeed feeding on the energy.

### Here's a multi-faceted approach to resolve your flickering portal and exploding lights:

1. **Gather materials**: Collect a lighter, a compass, and something personal to you. Arrange these items in a triangle while focusing on your portal issue.

2. **Soundwave manipulation**: Play "Running Up That Hill" backwards at the site of the portal. This should create a unique temporal resonance, addressing the flickering phenomenon.

3. **Stabilization**: Draw a salt circle and place the compass in the center. This will help stabilize any magnetic drift contributing to your problem.

Try these methods, and keep an eye on the emotional atmosphere in

In [11]:
#Step 4 – Test with Sample Complaints (v1 Compatible)
# Sample complaints
complaints = [
    "Why do demogorgons sometimes eat people and sometimes don't?",
    "The portal opens on different days—is there a schedule?",
    "Why can some psychics see the Downside Up and others can't?",
    "Why do creatures and power lines react so strangely together?",
]

def handle_complaint(complaint: str) -> str:
    """Handle a single complaint"""
    print(f"\n{'='*60}")
    print(f"COMPLAINT: {complaint}")
    print(f"{'='*60}\n")
    
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": complaint}
        ]
    })
    
    return result["messages"][-1].content


# Test with first 2 complaints
print("Testing agent with sample complaints...\n")

for complaint in complaints[:2]:
    response = handle_complaint(complaint)
    print(f"\nRESPONSE:\n{response}\n")

Testing agent with sample complaints...


COMPLAINT: Why do demogorgons sometimes eat people and sometimes don't?


RESPONSE:
The Demogorgon's perspective on why it sometimes eats people and sometimes doesn't is wrapped in the chaos of the Upside Down. It questions the assumptions of linear reasoning, suggesting that perhaps our understanding of its behavior doesn't fit neatly into our world’s logic. 

From Hawkins historical records, there's no direct documentation on Demogorgon behavior, but it does note that unexplained events are common in the area. Investigators are urged to consider factors like electromagnetic interference or recent activity near the gates to the Upside Down, which might influence the Demogorgon's actions.

As for the wisdom from our D&D party, they suggest that context is crucial. Mike emphasizes the need for specific details, while Dustin points out the importance of time, location, and recent changes in the environment. Lucas encourages looking for patterns, 

In [ ]:
# Step 5: Track tool calls with a Callback Handler - 5A) Tracker + Callback
#Analyze Agent Behavior - Understand how the agent uses tools creatively.

from langchain_core.callbacks import BaseCallbackHandler

class ToolUsageTracker:
    def __init__(self, tools):
        self.usage_count = {t.name: 0 for t in tools}
        self.tool_sequences = []

    def track_usage(self, tool_name: str):
        if tool_name in self.usage_count:
            self.usage_count[tool_name] += 1
            self.tool_sequences.append(tool_name)

    def get_statistics(self):
        total = sum(self.usage_count.values())
        most_used = max(self.usage_count.items(), key=lambda x: x[1])[0] if total > 0 else None
        return {
            "total_tool_calls": total,
            "tool_counts": self.usage_count,
            "most_used": most_used,
            "tool_sequences": self.tool_sequences
        }

class ToolTrackingCallback(BaseCallbackHandler):
    def __init__(self, tracker: ToolUsageTracker):
        self.tracker = tracker

    def on_tool_start(self, serialized, input_str=None, **kwargs):
        # serialized usually contains the tool name
        name = None
        if isinstance(serialized, dict):
            name = serialized.get("name") or serialized.get("id")
        if name:
            self.tracker.track_usage(name)

In [ ]:
# Step 5B) Run complaints WITH callbacks
tracker = ToolUsageTracker(tools)
cb = ToolTrackingCallback(tracker)

for complaint in complaints:
    _ = agent.invoke(
        {"messages": [{"role": "user", "content": complaint}]},
        config={"callbacks": [cb]}
    )

In [15]:
stats = tracker.get_statistics()

print("\n=== Tool Usage Analysis ===")
print(f"Total tool calls: {stats['total_tool_calls']}")
print(f"Tool usage counts: {stats['tool_counts']}")
print(f"Most used tool: {stats['most_used']}")

print("\nTool sequence (first 15 calls):")
print(" -> ".join(stats["tool_sequences"][:15]))


=== Tool Usage Analysis ===
Total tool calls: 9
Tool usage counts: {'consult_demogorgon': 4, 'check_hawkins_records': 4, 'cast_interdimensional_spell': 0, 'gather_party_wisdom': 1}
Most used tool: consult_demogorgon

Tool sequence (first 15 calls):
check_hawkins_records -> consult_demogorgon -> gather_party_wisdom -> check_hawkins_records -> consult_demogorgon -> consult_demogorgon -> check_hawkins_records -> check_hawkins_records -> consult_demogorgon
